# Activity gene summary


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import expit
from scipy.sparse import issparse
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import warnings
warnings.filterwarnings('ignore')

H5AD_PATH = "data/adata_cohort1.h5ad"
PERMUTATION_CSV = "results/permutation_full/high_vs_low_mean/gene_permutation_pvalues_raw.csv"
OUTPUT_CSV = "results/permutation_full/high_vs_low_mean/highlow_mean_gene_level_clean_summary.csv"

ZSCORE_LAYER = "zscores_4andhalf"
DONOR_COL = "unique_patient_id"
SLEDAI_COL = "sledai_score"
SLEDAI_HIGH_THRESHOLD = 11
SLEDAI_LOW_VALUE = 0
ZSCORE_THRESHOLD = 4.5

MAX_ITER = 25
TOL = 1e-8
PROB_CLIP = 1e-10

STATS_TO_INCLUDE = ['fisher', 'max_absZ']

KEY_GENES = ['SNRNP70', 'SSB', 'NPIPB15', 'CD2']

def vectorized_logistic_regression_full(X, Y):
    n_samples, n_features = X.shape

    p_mean = Y.mean()
    beta0 = np.full(n_features, np.log(p_mean / (1 - p_mean)), dtype=np.float32)
    beta1 = np.zeros(n_features, dtype=np.float32)
    Y_col = Y.reshape(-1, 1).astype(np.float32)

    for _ in range(MAX_ITER):
        eta = beta0 + beta1 * X
        p = expit(np.clip(eta, -500, 500))
        p = np.clip(p, PROB_CLIP, 1 - PROB_CLIP)

        W = p * (1 - p)
        r = Y_col - p

        H00 = W.sum(axis=0)
        H01 = (W * X).sum(axis=0)
        H11 = (W * X * X).sum(axis=0)
        U0 = r.sum(axis=0)
        U1 = (r * X).sum(axis=0)

        det = H00 * H11 - H01 * H01
        det = np.where(np.abs(det) < 1e-10, 1e-10, det)

        delta_beta0 = np.clip((H11 * U0 - H01 * U1) / det, -10, 10)
        delta_beta1 = np.clip((H00 * U1 - H01 * U0) / det, -10, 10)

        beta0 += delta_beta0
        beta1 += delta_beta1

        if np.max(np.maximum(np.abs(delta_beta0), np.abs(delta_beta1))) < TOL:
            break

    eta = beta0 + beta1 * X
    p = expit(np.clip(eta, -500, 500))
    p = np.clip(p, PROB_CLIP, 1 - PROB_CLIP)
    W = p * (1 - p)

    H00 = W.sum(axis=0)
    H01 = (W * X).sum(axis=0)
    H11 = (W * X * X).sum(axis=0)
    det = H00 * H11 - H01 * H01
    det = np.where(np.abs(det) < 1e-10, 1e-10, det)

    var_beta1 = H00 / det
    var_beta1 = np.where(var_beta1 < 0, np.nan, var_beta1)
    se = np.sqrt(var_beta1)

    z_scores = beta1 / se
    p_values = 2 * stats.norm.sf(np.abs(z_scores))

    return beta1.astype(np.float64), se.astype(np.float64), z_scores.astype(np.float64), p_values

def main():

    print("[1] Loading AnnData...")
    adata = sc.read_h5ad(H5AD_PATH)
    obs_df = adata.obs.copy()
    print(f"    Shape: {adata.shape}")
    print(f"    Donors: {obs_df[DONOR_COL].nunique()}, SLEDAI range: [{obs_df[SLEDAI_COL].min()}, {obs_df[SLEDAI_COL].max()}]")

    high_mask = obs_df[SLEDAI_COL] >= SLEDAI_HIGH_THRESHOLD
    low_mask = obs_df[SLEDAI_COL] == SLEDAI_LOW_VALUE

    donors_high = set(obs_df[high_mask][DONOR_COL].unique())
    donors_low = set(obs_df[low_mask][DONOR_COL].unique())
    donors_both = donors_high & donors_low
    donors_high_only = donors_high - donors_both
    donors_low_only = donors_low - donors_both

    keep_mask = (
        (high_mask & obs_df[DONOR_COL].isin(donors_high_only)) |
        (low_mask & obs_df[DONOR_COL].isin(donors_low_only))
    )
    adata_filtered = adata[keep_mask].copy()
    adata_filtered.obs['is_high'] = (adata_filtered.obs[SLEDAI_COL] >= SLEDAI_HIGH_THRESHOLD).astype(int)
    print(f"    After filtering: {adata_filtered.shape} (HIGH donors: {len(donors_high_only)}, LOW donors: {len(donors_low_only)}, OVERLAP donors: {len(donors_both)})")

    zscores = adata_filtered.layers[ZSCORE_LAYER]
    if issparse(zscores):
        zscores = zscores.toarray()
    peptide_ids = adata_filtered.var_names.tolist()
    gene_names = adata_filtered.var['gene'].astype(str).tolist()

    print("[2] Aggregating to donor level (MEAN)...")
    obs_filt = adata_filtered.obs.copy()
    obs_filt['sample_idx'] = np.arange(len(obs_filt))
    obs_filt[DONOR_COL] = obs_filt[DONOR_COL].astype(str)

    all_donors = sorted(set(donors_high_only) | set(donors_low_only))
    n_peptides = len(peptide_ids)
    Z_donor = np.zeros((len(all_donors), n_peptides), dtype=np.float32)
    Y = np.zeros(len(all_donors), dtype=np.int32)

    for i, donor in enumerate(all_donors):
        donor_mask = obs_filt[DONOR_COL] == donor
        sample_idx = obs_filt.loc[donor_mask, 'sample_idx'].values
        Z_donor[i, :] = zscores[sample_idx, :].mean(axis=0)
        Y[i] = 1 if donor in donors_high_only else 0

    n_high = (Y == 1).sum()
    n_low = (Y == 0).sum()
    print(f"    Donors: {len(all_donors)} (HIGH={n_high}, LOW={n_low})")

    del adata, adata_filtered, zscores

    print("[3] Running logistic regression on all peptides...")
    log_or, se, z_scores, p_values = vectorized_logistic_regression_full(Z_donor, Y)
    print(f"    Done. z-score range: [{z_scores[np.isfinite(z_scores)].min():.2f}, "
          f"{z_scores[np.isfinite(z_scores)].max():.2f}]")

    print("[4] Computing per-peptide discrete proportions (z >= 4.5)...")
    high_reactivity = (Z_donor >= ZSCORE_THRESHOLD)
    prop_high_case = high_reactivity[Y == 1, :].mean(axis=0)
    prop_high_ctrl = high_reactivity[Y == 0, :].mean(axis=0)

    print("[5] Building peptide-level DataFrame...")
    mapping = pd.DataFrame({'seq_id': peptide_ids, 'gene': gene_names})

    peptide_df = pd.DataFrame({
        'peptide': peptide_ids,
        'log_or': log_or,
        'se': se,
        'z_score': z_scores,
        'p_value': p_values,
        'odds_ratio': np.exp(log_or),
        'prop_high_case_donors': prop_high_case,
        'prop_high_control_donors': prop_high_ctrl,
    })

    merge_cols = ['seq_id', 'gene']
    if 'pep_short' in mapping.columns:
        merge_cols.append('pep_short')
    peptide_df = peptide_df.merge(
        mapping[merge_cols], left_on='peptide', right_on='seq_id', how='left'
    )
    peptide_df = peptide_df.dropna(subset=['gene', 'p_value'])
    print(f"    Peptides with gene annotation: {len(peptide_df):,}")

    print("[6] Aggregating to gene level (min-p peptide per gene)...")
    idx_min = peptide_df.groupby('gene')['p_value'].idxmin()
    gene_effects = peptide_df.loc[idx_min].copy()

    n_pep_per_gene = peptide_df.groupby('gene').size().rename('n_peptides')
    gene_effects = gene_effects.merge(n_pep_per_gene, left_on='gene', right_index=True, how='left')

    gene_effects = gene_effects.rename(columns={
        'peptide': 'best_peptide',
        'p_value': 'best_peptide_pval',
    })
    if 'pep_short' in gene_effects.columns:
        gene_effects = gene_effects.rename(columns={'pep_short': 'best_pep_short'})

    cols_to_keep = [
        'gene', 'n_peptides', 'best_pep_short', 'best_peptide',
        'log_or', 'z_score', 'odds_ratio', 'best_peptide_pval',
        'prop_high_case_donors', 'prop_high_control_donors',
    ]
    cols_to_keep = [c for c in cols_to_keep if c in gene_effects.columns]
    gene_effects = gene_effects[cols_to_keep].reset_index(drop=True)
    print(f"    {len(gene_effects):,} genes")

    print("[7] Loading permutation results...")
    perm_df = pd.read_csv(PERMUTATION_CSV)
    print(f"    {len(perm_df):,} genes with permutation p-values")

    print("[8] Merging...")
    merged = gene_effects.merge(perm_df, on='gene', how='inner')
    print(f"    {len(merged):,} genes after merge")

    final_cols = [
        'gene', 'n_peptides', 'best_pep_short', 'best_peptide',
        'log_or', 'z_score', 'odds_ratio', 'best_peptide_pval',
        'prop_high_case_donors', 'prop_high_control_donors',
    ]

    for stat in STATS_TO_INCLUDE:
        for suffix in ['_obs', '_perm_p']:
            col = f'{stat}{suffix}'
            if col in merged.columns:
                final_cols.append(col)

    print("[9] Computing BH FDR q-values...")
    for stat in STATS_TO_INCLUDE:
        perm_col = f'{stat}_perm_p'
        qval_col = f'{stat}_qval'
        if perm_col in merged.columns:
            pvals = merged[perm_col].values
            mask = np.isfinite(pvals)
            merged[qval_col] = np.nan
            if mask.sum() > 0:
                _, qvals, _, _ = multipletests(pvals[mask], method='fdr_bh')
                merged.loc[merged.index[mask], qval_col] = qvals
            final_cols.append(qval_col)

            n_sig_01 = (merged[qval_col] <= 0.1).sum()
            n_sig_005 = (merged[qval_col] <= 0.05).sum()
            print(f"    {stat}: FDR<0.1 = {n_sig_01}, FDR<0.05 = {n_sig_005}")

    final_cols = [c for c in final_cols if c in merged.columns]
    clean_df = merged[final_cols].copy()

    sort_col = 'truncsum_0010_perm_p' if 'truncsum_0010_perm_p' in clean_df.columns else 'fisher_perm_p'
    clean_df = clean_df.sort_values(sort_col).reset_index(drop=True)

    print(f"\n[10] Saving to {OUTPUT_CSV}")
    clean_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nShape: {clean_df.shape}")
    print(f"\nColumns ({len(clean_df.columns)}):")
    for col in clean_df.columns:
        print(f"  - {col}")

    print(f"\nTop 15 genes by {sort_col}:")
    preview_cols = ['gene', 'best_pep_short', 'n_peptides', 'log_or', 'z_score',
                    'truncsum_0010_perm_p', 'truncsum_0010_qval',
                    'fisher_perm_p', 'fisher_qval',
                    'max_absZ_perm_p', 'max_absZ_qval']
    preview_cols = [c for c in preview_cols if c in clean_df.columns]
    print(clean_df[preview_cols].head(15).to_string(index=False))

    key_df = clean_df[clean_df['gene'].isin(KEY_GENES)]
    if len(key_df) > 0:
        print(f"\nKey genes:")
        print(key_df[preview_cols].to_string(index=False))

    return clean_df

if __name__ == '__main__':
    clean_df = main()